# Partitioning articles by topic

## Import statements

In [1]:
from bertopic import BERTopic
import pandas as pd
import os
import numpy as np

/Users/ameliemajor/miniforge3/envs/bertopic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import plotly.io as pio
pio.renderers.default = "browser"


## Load dataframe and filter by `bodyContent`

In [3]:
political_guardian_articles_df = pd.read_csv('data/political_guardian_articles.csv')
political_guardian_articles_df.head()
df = political_guardian_articles_df.copy()

In [4]:
df = political_guardian_articles_df

# Keep ONLY rows with real text for BERTopic (preserve index for safe merge-back)
mask = (
    df["bodyContent"].notna()
    & df["bodyContent"].astype(str).str.strip().ne("")
    & df["bodyContent"].astype(str).str.lower().ne("nan")
)

df_fit = df.loc[mask].copy()
bodyText = df_fit["bodyContent"].astype(str).tolist()

print(f"Full DataFrame shape: {df.shape}")
print(f"Rows used for BERTopic: {df_fit.shape}")
print(f"Texts passed to BERTopic: {len(bodyText)}")
print(f"Dropped rows: {df.shape[0] - df_fit.shape[0]}")


Full DataFrame shape: (13553, 7)
Rows used for BERTopic: (13526, 7)
Texts passed to BERTopic: 13526
Dropped rows: 27


## Fit BERTopic to `bodyContent`

In [5]:
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2")
print("Fitting the BERTopic model...")
topics, probs = topic_model.fit_transform(bodyText)
df["topic_id"] = np.nan
df.loc[df_fit.index, "topic_id"] = topics
df["topic_id"] = df["topic_id"].astype("Int64")

Fitting the BERTopic model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1165.28it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Visualize findings

In [6]:
topic_model.visualize_barchart()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.get_topic_info(10)

## Select non-person-entity topics
e.g. not a politician as a topic

In [7]:
# topic_id has already been assigned safely using the fitted subset's index.
# (Do NOT assign topic_model.topics_ directly to the full dataframe; lengths can differ.)
assert "topic_id" in df.columns, "topic_id column missing—run the BERTopic fitting cell first."
print("topic_id assigned:", df["topic_id"].notna().sum(), "rows; missing:", df["topic_id"].isna().sum())
political_guardian_articles_df = df


topic_id assigned: 13526 rows; missing: 27


## Write to files

In [8]:
topic_index = {
    0: "US Politics",
    1: "Covid-19",
    2: "Brexit Referendum",
    3: "Labour Party",
    4: "Russia-Ukraine Conflict"
}

df["topic_name"] = df["topic_id"].map(topic_index).fillna("Other")

os.makedirs("articles_by_topic", exist_ok=True)

for topic_id, topic_name in topic_index.items():
    subset = df[df.topic_id == topic_id]
    subset.to_csv(
        f"articles_by_topic/{topic_name.replace(' ', '_')}.csv",
        index=False
    )

df[df["topic_name"] == "Other"].to_csv("articles_by_topic/Other.csv", index=False)


political_guardian_articles_df = df
